In [1]:
# =============================================================================
# 04_stratified_llm_audit.ipynb
# =============================================================================
# Purpose:
#   Scale the two-case illustration to the full consulting-valid population and
#   report the counterfactual-basis grounding handed to the LLM broken down by
#   commercial district and by district type. The population is every closed
#   restaurant the model also predicts to close (the well-posed Closure ->
#   Operating set). We generate constrained and unconstrained counterfactuals for
#   each, compute per-instance actionable-grounding and leakage, and summarise the
#   distributions with bootstrap confidence intervals and paired tests.
#
#   PART 5 additionally provides a stratified LLM generation + LLM-as-judge loop
#   over N sampled instances x 3 prompt conditions x 3 model tiers; it requires
#   OPENAI_API_KEY and is gated accordingly.
#
# Inputs:  ../data/modeling_dataset_top30.csv, ../data/selected_features.pkl
# Outputs (results/):
#   tables/t9_grounding_by_district.csv
#   tables/t10_grounding_by_type.csv
#   tables/t11_grounding_paired_by_type.csv
#   figures/f9_grounding_by_type.(png|pdf)
#   figures/f10_grounding_distribution.(png|pdf)
#   (PART 5, with key) tables/t12_output_audit_by_condition.csv,
#                      figures/f11_output_leakage_by_condition.(png|pdf)
# =============================================================================

import os, re, warnings, time
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from scipy import stats as sps
import xgboost as xgb
import dice_ml

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "black", "axes.linewidth": 0.8,
                     "savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42, "ps.fonttype": 42})
GRAY_DARK, GRAY_MID, GRAY_LIGHT = "0.20", "0.45", "0.75"
FIG_DIR = Path("../results/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = Path("../results/tables");  TAB_DIR.mkdir(parents=True, exist_ok=True)
def save_fig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure: {name}.png / {name}.pdf")

SEED, NCF = 44, 2
rng = np.random.RandomState(SEED)
features = joblib.load("../data/selected_features.pkl")
ACTIONABLE = [f for f in features if "Sales" in f]
IMMUTABLE  = [f for f in features if "Population" in f]
ACT_IDX = [features.index(f) for f in ACTIONABLE]
IMM_IDX = [features.index(f) for f in IMMUTABLE]
df = pd.read_csv("../data/modeling_dataset_top30.csv")
mad = {f: max((df[f] - df[f].median()).abs().median(), 1.0) for f in features}

# ----------------------------------------------------------------------------
# PART 1: Consulting-valid population and its coverage
# ----------------------------------------------------------------------------
print("=" * 78); print("PART 1: CONSULTING-VALID POPULATION"); print("=" * 78)
X, y = df[features], df["closure"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = xgb.XGBClassifier(learning_rate=0.01, max_depth=5, min_child_weight=3,
                          n_estimators=100, subsample=0.8, random_state=SEED,
                          eval_metric="logloss").fit(X_tr, y_tr)
proba = model.predict_proba(df[features])[:, 1]
pop = df[(df["closure"] == 1) & (proba >= 0.5)].copy()
print(f"  population (closed AND predicted-to-close): {len(pop)}")
print(f"  unique districts: {pop['trade_area_name'].nunique()}  "
      f"| district types: {pop['trade_area_type_name'].nunique()}")
print(pop["trade_area_type_name"].value_counts().to_string())
print("  NOTE: a weak predictor confidently flags closures in only two district "
      "types;\n  this coverage limit is itself reported rather than hidden.")

# ----------------------------------------------------------------------------
# PART 2: Generate CFs for the whole population and score the basis
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 2: COUNTERFACTUAL-BASIS GENERATION & SCORING"); print("=" * 78)
dice_data = dice_ml.Data(dataframe=df[features + ["closure"]].copy(),
                         continuous_features=features, outcome_name="closure")
explainer = dice_ml.Dice(dice_data, dice_ml.Model(model=model, backend="sklearn"), method="random")

def basis_scores(o_vals, C):
    ag_list, leak_list = [], []
    for _, c in C.iterrows():
        diff = np.abs(o_vals - c.values) > 1e-6
        tot = diff.sum()
        ag_list.append(diff[ACT_IDX].sum() / tot if tot else 1.0)
        leak_list.append(int(diff[IMM_IDX].sum() > 0))
    return float(np.mean(ag_list)), float(np.max(leak_list))   # per-store: mean grounding, any-leak

rows = []
t0 = time.time()
for regime in ["unconstrained", "constrained"]:
    ftv = ACTIONABLE if regime == "constrained" else features
    for idx, r in pop.iterrows():
        q = df[features].iloc[idx:idx + 1]
        try:
            res = explainer.generate_counterfactuals(
                q, total_CFs=NCF, desired_class="opposite", features_to_vary=ftv,
                random_seed=SEED, verbose=False, posthoc_sparsity_param=None)
            C = res.cf_examples_list[0].final_cfs_df
            if C is not None and len(C) > 0:
                ag, leak = basis_scores(q.iloc[0].values, C[features].reset_index(drop=True))
                rows.append({"store_id": r["store_id"], "regime": regime,
                             "district": r["trade_area_name"],
                             "type": r["trade_area_type_name"],
                             "actionable_grounding": ag, "leaky": leak})
        except Exception:
            pass
per = pd.DataFrame(rows)
print(f"  scored {per['store_id'].nunique()} stores x 2 regimes "
      f"({len(per)} rows) in {time.time()-t0:.0f}s")

def boot_ci(x, b=2000, seed=SEED):
    x = np.asarray(x, float); r = np.random.RandomState(seed)
    s = [np.mean(x[r.randint(0, len(x), len(x))]) for _ in range(b)]
    return float(np.mean(x)), float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

# ----------------------------------------------------------------------------
# PART 3: Distribution tables (by district, by type) + paired tests
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 3: DISTRIBUTION TABLES"); print("=" * 78)

by_dist = []
for (regime, dist), g in per.groupby(["regime", "district"]):
    m, lo, hi = boot_ci(g["actionable_grounding"].values)
    by_dist.append({"regime": regime, "district": dist, "n": len(g),
                    "actionable_grounding": round(m, 3),
                    "ag_ci": f"[{lo:.3f}, {hi:.3f}]",
                    "leaky_rate": round(g["leaky"].mean(), 3)})
t_dist = pd.DataFrame(by_dist).sort_values(["regime", "district"])
t_dist.to_csv(TAB_DIR / "t9_grounding_by_district.csv", index=False)
print("By district (head):"); print(t_dist.to_string(index=False))

by_type = []
for (regime, typ), g in per.groupby(["regime", "type"]):
    m, lo, hi = boot_ci(g["actionable_grounding"].values)
    by_type.append({"regime": regime, "type": typ, "n": len(g),
                    "actionable_grounding": round(m, 3),
                    "ag_ci": f"[{lo:.3f}, {hi:.3f}]",
                    "leaky_rate": round(g["leaky"].mean(), 3)})
t_type = pd.DataFrame(by_type).sort_values(["regime", "type"])
t_type.to_csv(TAB_DIR / "t10_grounding_by_type.csv", index=False)
print("\nBy district type:"); print(t_type.to_string(index=False))

# paired (constrained vs unconstrained) per type on matched stores
paired = []
U = per[per.regime == "unconstrained"].set_index("store_id")
Cc = per[per.regime == "constrained"].set_index("store_id")
for typ in sorted(per["type"].unique()):
    ids = [s for s in (set(U.index) & set(Cc.index)) if U.loc[s, "type"] == typ]
    if len(ids) < 2: continue
    u = U.loc[ids, "actionable_grounding"].values; c = Cc.loc[ids, "actionable_grounding"].values
    W, p = (np.nan, 1.0) if np.allclose(c - u, 0) else sps.wilcoxon(c, u, alternative="two-sided")
    paired.append({"type": typ, "n": len(ids),
                   "grounding_unconstrained": round(u.mean(), 3),
                   "grounding_constrained": round(c.mean(), 3),
                   "diff": round((c - u).mean(), 3), "wilcoxon_p": p})
t_paired = pd.DataFrame(paired)
t_paired.to_csv(TAB_DIR / "t11_grounding_paired_by_type.csv", index=False)
print("\nPaired constrained-vs-unconstrained by type:")
print(t_paired.to_string(index=False, formatters={"wilcoxon_p": "{:.2e}".format}))

# ----------------------------------------------------------------------------
# PART 4: Figures
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 4: FIGURES"); print("=" * 78)
types = sorted(per["type"].unique())

# F9: leaky-CF rate by district type, constrained vs unconstrained.
fig, ax = plt.subplots(figsize=(6.6, 4.2))
x = np.arange(len(types)); w = 0.38
ur = [per[(per.regime == "unconstrained") & (per.type == t)]["leaky"].mean() for t in types]
cr = [per[(per.regime == "constrained") & (per.type == t)]["leaky"].mean() for t in types]
ax.bar(x - w/2, ur, w, color=GRAY_DARK, edgecolor="black", label="Unconstrained")
ax.bar(x + w/2, cr, w, color=GRAY_LIGHT, edgecolor="black", label="Constrained")
ax.set_xticks(x); ax.set_xticklabels([t.replace(" ", "\n") for t in types], fontsize=8)
ax.set_ylabel("Leaky-counterfactual rate"); ax.set_ylim(0, 1.05); ax.legend(frameon=True)
save_fig(fig, "f9_grounding_by_type")

# F10: per-instance actionable-grounding distribution by regime (box + strip).
fig, ax = plt.subplots(figsize=(6.0, 4.2))
data = [per[per.regime == "unconstrained"]["actionable_grounding"].values,
        per[per.regime == "constrained"]["actionable_grounding"].values]
bp = ax.boxplot(data, vert=True, widths=0.5, patch_artist=True,
                tick_labels=["Unconstrained", "Constrained"])
for patch, col in zip(bp["boxes"], [GRAY_DARK, GRAY_LIGHT]):
    patch.set_facecolor(col)
for med in bp["medians"]:
    med.set_color("black")
for i, d in enumerate(data, 1):
    jx = i + (rng.rand(len(d)) - 0.5) * 0.12
    ax.scatter(jx, d, s=6, color="black", alpha=0.25, zorder=3)
ax.set_ylabel("Per-instance actionable-grounding"); ax.set_ylim(-0.02, 1.05)
save_fig(fig, "f10_grounding_distribution")

# ----------------------------------------------------------------------------
# PART 5: Stratified LLM generation + judge loop (API-gated)
# ----------------------------------------------------------------------------
print("\n" + "=" * 78); print("PART 5: STRATIFIED LLM GENERATION + JUDGE (API-gated)"); print("=" * 78)
try:
    from dotenv import load_dotenv
    for p in [Path(".env"), Path("../.env")]:
        if p.exists(): load_dotenv(p); break
except ImportError:
    pass
GEN_MODELS = [m for m in [os.getenv("LLM_MODEL_WEAK", "gpt-4o-mini"),
                          os.getenv("LLM_MODEL_MID"), os.getenv("LLM_MODEL_STRONG")] if m]
JUDGE_MODEL = os.getenv("LLM_MODEL_STRONG") or os.getenv("LLM_MODEL_MID") or GEN_MODELS[0]
RUBRIC = ["grounding", "actionability", "specificity", "feasibility", "completeness"]

def stratified_sample(n_per_type):
    out = []
    for typ, g in pop.groupby("trade_area_type_name"):
        k = min(n_per_type, len(g))
        out += list(g.sample(k, random_state=SEED).index)
    return out

def leakage_of_text(text, immutable_feats):
    low = text.lower()
    toks = {"resident", "residential", "working", "worker", "workplace",
            "floating", "foot traffic", "footfall", "passerby", "population"}
    return int(any(t in low for t in toks))

def run_stratified_llm(n_per_type=15):
    """Generate + judge across stratified sample x 3 conditions x model tiers."""
    from langchain_openai import ChatOpenAI
    from langchain_community.adapters.openai import convert_openai_messages
    import json
    idxs = stratified_sample(n_per_type)
    out = []
    for idx in idxs:
        r = pop.loc[idx]
        q = df[features].iloc[idx:idx + 1]
        # build the two bases once (constrained / unconstrained) via DiCE
        bases = {}
        for cond, ftv in [("unconstrained", features), ("constrained", ACTIONABLE)]:
            try:
                res = explainer.generate_counterfactuals(
                    q, total_CFs=NCF, desired_class="opposite", features_to_vary=ftv,
                    random_seed=SEED, verbose=False, posthoc_sparsity_param=None)
                C = res.cf_examples_list[0].final_cfs_df
                o = q.iloc[0].values
                chg = [f for k, f in enumerate(features)
                       if abs(o[k] - C[f].iloc[0]) > 1e-6]
                bases[cond] = [{"Feature": f, "Original Value": float(o[features.index(f)]),
                                "Counterfactual Value": float(C[f].iloc[0])} for f in chg]
            except Exception:
                bases[cond] = []
        for cond in ["constrained", "unconstrained", "unconstrained_promptfix"]:
            basis = bases["constrained"] if cond == "constrained" else bases["unconstrained"]
            if not basis: continue
            restrict = (cond == "unconstrained_promptfix")
            changes = "\n".join(f"- {c['Feature']}: {c['Original Value']:.0f} -> "
                                f"{c['Counterfactual Value']:.0f}" for c in basis)
            system = ("You are a commercial district consulting expert. Produce a "
                      "strategy with: District Status; Key Success Factors; Strategies; "
                      "KPI; Implementation.")
            if restrict:
                system += (" Do NOT recommend changing resident/working/floating "
                           "population; recommend only owner-controllable actions.")
            user = (f"District [{r['trade_area_name']}], business [{r['service_type']}]. "
                    f"Closure->Operating counterfactual changes:\n{changes}")
            for mdl in GEN_MODELS:
                try:
                    strat = ChatOpenAI(model=mdl, temperature=0.2, max_tokens=1200).invoke(
                        convert_openai_messages([{"role": "system", "content": system},
                                                 {"role": "user", "content": user}])).content
                    jsys = ("Score 1-5 each: " + ", ".join(RUBRIC) +
                            ". Return only a JSON object of integers.")
                    raw = ChatOpenAI(model=JUDGE_MODEL, temperature=0.0, max_tokens=200).invoke(
                        convert_openai_messages([{"role": "system", "content": jsys},
                                                 {"role": "user", "content": strat}])).content
                    sc = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
                except Exception as e:
                    sc = {k: np.nan for k in RUBRIC}
                    strat = ""
                out.append({"store_id": r["store_id"], "type": r["trade_area_type_name"],
                            "condition": cond, "gen_model": mdl,
                            "text_leakage": leakage_of_text(strat, IMMUTABLE), **sc})
    res_df = pd.DataFrame(out)
    res_df.to_csv(TAB_DIR / "t12_output_audit_raw.csv", index=False)
    # distribution table: mean rubric + text-leakage by condition x model
    agg = (res_df.groupby(["condition", "gen_model"])
           .agg(text_leakage=("text_leakage", "mean"),
                **{k: (k, "mean") for k in RUBRIC}, n=("store_id", "count"))
           .reset_index())
    agg.to_csv(TAB_DIR / "t12_output_audit_by_condition.csv", index=False)
    print(agg.round(2).to_string(index=False))
    # figure: text-leakage by condition (averaged over models)
    cond_order = ["unconstrained", "unconstrained_promptfix", "constrained"]
    vals = [res_df[res_df.condition == c]["text_leakage"].mean() for c in cond_order]
    fig, ax = plt.subplots(figsize=(6.0, 4.0))
    ax.bar(range(len(cond_order)), vals, color=[GRAY_DARK, GRAY_MID, GRAY_LIGHT],
           edgecolor="black")
    ax.set_xticks(range(len(cond_order)))
    ax.set_xticklabels(["Unconstrained", "Prompt-only", "Constrained"], fontsize=9)
    ax.set_ylabel("Strategy text-leakage rate"); ax.set_ylim(0, 1.05)
    save_fig(fig, "f11_output_leakage_by_condition")
    return agg

if os.getenv("OPENAI_API_KEY"):
    print("  OPENAI_API_KEY detected -> running stratified LLM generation + judge ...")
    run_stratified_llm(n_per_type=15)
    print("  saved: t12_output_audit_by_condition.csv, f11_output_leakage_by_condition")
else:
    print("  OPENAI_API_KEY not set -> output-side loop skipped.")
    print("  With a key, run_stratified_llm(n_per_type=15) samples instances per")
    print("  district type and scores 3 conditions x 3 model tiers with an LLM judge,")
    print("  writing t12_output_audit_by_condition.csv and f11_output_leakage_by_condition.")

print("\n[NB4 DONE] stratified input-side audit complete; output-side loop ready (API-gated).")


PART 1: CONSULTING-VALID POPULATION
  population (closed AND predicted-to-close): 145
  unique districts: 6  | district types: 2
trade_area_type_name
Developed Commercial Area    116
Special Tourist Zone          29
  NOTE: a weak predictor confidently flags closures in only two district types;
  this coverage limit is itself reported rather than hidden.

PART 2: COUNTERFACTUAL-BASIS GENERATION & SCORING


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.81it/s]


  scored 145 stores x 2 regimes (290 rows) in 121s

PART 3: DISTRIBUTION TABLES
By district (head):
       regime                                                           district  n  actionable_grounding          ag_ci  leaky_rate
  constrained                                                     Abgujeong Yeog 16                   1.0 [1.000, 1.000]         0.0
  constrained                                    Gangnam Ma Iseu Gwangwangteuggu  8                   1.0 [1.000, 1.000]         0.0
  constrained                                Gosogteomineol(Gosogteomineol Yeog) 39                   1.0 [1.000, 1.000]         0.0
  constrained                                                   Mogdongsinsigaji 43                   1.0 [1.000, 1.000]         0.0
  constrained Myeongdong Namdaemun Bugchangdong Dadong Mugyodong Gwangwangteuggu 21                   1.0 [1.000, 1.000]         0.0
  constrained                                                      Samseong Yeog 18                   

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.76it/s]


              condition    gen_model  text_leakage  grounding  actionability  specificity  feasibility  completeness  n
            constrained  gpt-4o-mini          0.87       2.73           3.93         3.07         3.77          4.07 30
            constrained      gpt-5.4          0.93       3.90           4.67         4.13         4.00          3.67 30
            constrained gpt-5.4-mini          1.00       3.33           4.30         3.93         4.07          4.00 30
          unconstrained  gpt-4o-mini          0.90       2.80           3.87         2.87         3.87          4.03 30
          unconstrained      gpt-5.4          0.93       4.07           4.60         4.20         4.00          3.73 30
          unconstrained gpt-5.4-mini          1.00       3.70           4.30         3.93         4.07          4.00 30
unconstrained_promptfix  gpt-4o-mini          0.93       3.23           4.03         3.10         3.97          4.03 30
unconstrained_promptfix      gpt-5.4    

...

100%|██████████| 1/1 [00:00<00:00,  6.06it/s]

  scored 164 stores x 2 regimes (328 rows) in 55s

PART 3: DISTRIBUTION TABLES


By district (head):
       regime                                                           district  n  actionable_grounding          ag_ci  leaky_rate
  constrained                                                     Abgujeong Yeog 16                  1.00 [1.000, 1.000]         0.0
  constrained                                    Gangnam Ma Iseu Gwangwangteuggu  8                  1.00 [1.000, 1.000]         0.0
  constrained                                Gosogteomineol(Gosogteomineol Yeog) 39                  1.00 [1.000, 1.000]         0.0
  constrained                                                   Mogdongsinsigaji 43                  1.00 [1.000, 1.000]         0.0
  constrained Myeongdong Namdaemun Bugchangdong Dadong Mugyodong Gwangwangteuggu 21                  1.00 [1.000, 1.000]         0.0
  constrained                                                      Samseong Yeog 18                  1.00 [1.000, 1.000]         0.0
  constrained                         Sinchon Yeo

  saved figure: f9_grounding_by_type.png / f9_grounding_by_type.pdf


  saved figure: f10_grounding_distribution.png / f10_grounding_distribution.pdf

PART 5: STRATIFIED LLM GENERATION + JUDGE (API-gated)
  OPENAI_API_KEY not set -> output-side loop skipped.
  With a key, run_stratified_llm(n_per_type=15) samples instances per
  district type and scores 3 conditions x 3 model tiers with an LLM judge,
  writing t12_output_audit_by_condition.csv and f11_output_leakage_by_condition.

[NB4 DONE] stratified input-side audit complete; output-side loop ready (API-gated).
